# Industrial Readiness Evidence

This notebook exercises SC-NeuroCore's industrial application registry and evidence-readiness checks.

## Evidence Boundary

This notebook evaluates local readiness profiles only. It does not claim that aerospace, automotive, medical, rail, or industrial-control certification has been achieved. Certification requires external evidence packs, independent review, target hardware results, traceability matrices, and authority-specific safety cases.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

from sc_neurocore.industrial_applications import (
    EvidenceCategory,
    IndustrialApplicationRegistry,
    IndustrialDomain,
    assess_industrial_readiness,
)
from sc_neurocore.safety_cert import EvidenceBag, EvidenceItem

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent

registry = IndustrialApplicationRegistry()
profiles = registry.list_profiles()
profile_summary = [
    {
        "domain": profile.domain.value,
        "name": profile.name,
        "standards": [standard.value for standard in profile.safety_standards],
        "target_sil": profile.target_sil.name if profile.target_sil else None,
        "target_asil": profile.target_asil.value if profile.target_asil else None,
        "hazard_count": len(profile.hazards),
        "required_module_count": len(profile.required_modules),
        "mandatory_evidence_categories": [
            requirement.category.value
            for requirement in profile.evidence_requirements
            if requirement.mandatory
        ],
    }
    for profile in profiles
]

assert {item["domain"] for item in profile_summary} == {domain.value for domain in IndustrialDomain}
assert all(item["hazard_count"] > 0 for item in profile_summary)
profile_summary

In [ ]:
category_counts = Counter(
    requirement.category.value
    for profile in profiles
    for requirement in profile.evidence_requirements
    if requirement.mandatory
)
standard_counts = Counter(
    standard.value
    for profile in profiles
    for standard in profile.safety_standards
)
coverage_matrix = {
    "mandatory_category_counts": dict(sorted(category_counts.items())),
    "standard_counts": dict(sorted(standard_counts.items())),
    "domains": [profile.domain.value for profile in profiles],
}
assert coverage_matrix["mandatory_category_counts"]["test"] == len(profiles)
assert "IEC 61508" in coverage_matrix["standard_counts"]
coverage_matrix

In [ ]:
def evidence_bag(*categories: str) -> EvidenceBag:
    bag = EvidenceBag()
    for index, category in enumerate(categories):
        bag.add(
            EvidenceItem(
                filename=f"local_evidence_{index}.json",
                category=category,
                description=f"local {category} evidence placeholder for readiness arithmetic",
            )
        )
    return bag

partial_aerospace = assess_industrial_readiness(
    IndustrialDomain.AEROSPACE,
    evidence_bag("design", "formal"),
)
partial_summary = partial_aerospace.to_dict()
missing_categories = {item["category"] for item in partial_summary["missing_mandatory"]}

assert partial_summary["ready"] is False
assert "test" in missing_categories
assert "hil" in missing_categories
partial_summary

In [ ]:
complete_industrial_control = assess_industrial_readiness(
    "industrial_control",
    evidence_bag("design", "test", "analysis", "report"),
)
complete_medical_alias = assess_industrial_readiness(
    IndustrialDomain.MEDICAL,
    evidence_bag("design", "test", "analysis", "hardware-in-loop", "report", "security"),
)
complete_summary = {
    "industrial_control_ready": complete_industrial_control.ready,
    "industrial_control_coverage": complete_industrial_control.mandatory_coverage,
    "medical_alias_ready": complete_medical_alias.ready,
    "medical_present_categories": [category.value for category in complete_medical_alias.present_categories],
}

assert complete_summary["industrial_control_ready"] is True
assert complete_summary["industrial_control_coverage"] == 1.0
assert EvidenceCategory.HIL.value in complete_summary["medical_present_categories"]
complete_summary

In [ ]:
domain_rollup = {}
for profile in profiles:
    empty_assessment = registry.assess(profile.domain, EvidenceBag())
    domain_rollup[profile.domain.value] = {
        "ready_without_evidence": empty_assessment.ready,
        "coverage_without_evidence": empty_assessment.mandatory_coverage,
        "missing_mandatory_count": len(empty_assessment.missing_mandatory),
    }

assert all(item["ready_without_evidence"] is False for item in domain_rollup.values())
assert all(item["coverage_without_evidence"] == 0.0 for item in domain_rollup.values())
domain_rollup

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.industrial-readiness-evidence.v1",
    "profile_summary": profile_summary,
    "coverage_matrix": coverage_matrix,
    "partial_aerospace": {
        "ready": partial_summary["ready"],
        "mandatory_coverage": partial_summary["mandatory_coverage"],
        "missing_categories": sorted(missing_categories),
    },
    "complete_examples": complete_summary,
    "fail_closed_rollup": domain_rollup,
    "evidence_boundary": "Readiness arithmetic only; no certification, target-hardware, or authority-accepted safety-case claim.",
}
manifest